<a href="https://colab.research.google.com/github/deepakawl/supplychain-analytics-teaching/blob/main/Demand_Forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

iPhone Sales Demand Forecasting - Advanced Methods
====================================================
This script demonstrates multiple forecasting approaches on iPhone quarterly sales data,
covering: Basic methods (Naive, MA, Exp Smoothing), Regression, ARIMA-style,
and Machine Learning (Random Forest, Neural Net via sklearn).

Author: Deepak Agrawal, www.dagrawal.com
Prepared for SCM Demand Forecasting Class
Data: iPhone Unit Sales (Quarterly), 2019-2024

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

## DATA SETUP

In [2]:
data = {
    'Quarter': ['1Q19','2Q19','3Q19','4Q19','1Q20','2Q20','3Q20','4Q20',
                '1Q21','2Q21','3Q21','4Q21','1Q22','2Q22','3Q22','4Q22',
                '1Q23','2Q23','3Q23','4Q23','1Q24','2Q24','3Q24','4Q24'],
    'FiscalQtr': [1,2,3,4]*6,
    'Sales': [37.04,35.06,26.03,26.91,47.79,37.43,31.24,33.80,
              51.03,43.72,35.20,39.27,74.47,61.17,47.53,48.05,
              74.78,51.19,40.40,45.51,78.29,50.76,41.03,46.68]
}

In [3]:
df = pd.DataFrame(data)
df['Period'] = range(1, len(df)+1)
df['Year'] = [2019]*4 + [2020]*4 + [2021]*4 + [2022]*4 + [2023]*4 + [2024]*4

print("="*70)
print("iPhone Quarterly Unit Sales Data (in Millions)")
print("="*70)
print(df[['Quarter','FiscalQtr','Sales']].to_string(index=False))
print(f"\nTotal observations: {len(df)}")
print(f"Mean: {df['Sales'].mean():.2f} MM | Std: {df['Sales'].std():.2f} MM")

iPhone Quarterly Unit Sales Data (in Millions)
Quarter  FiscalQtr  Sales
   1Q19          1  37.04
   2Q19          2  35.06
   3Q19          3  26.03
   4Q19          4  26.91
   1Q20          1  47.79
   2Q20          2  37.43
   3Q20          3  31.24
   4Q20          4  33.80
   1Q21          1  51.03
   2Q21          2  43.72
   3Q21          3  35.20
   4Q21          4  39.27
   1Q22          1  74.47
   2Q22          2  61.17
   3Q22          3  47.53
   4Q22          4  48.05
   1Q23          1  74.78
   2Q23          2  51.19
   3Q23          3  40.40
   4Q23          4  45.51
   1Q24          1  78.29
   2Q24          2  50.76
   3Q24          3  41.03
   4Q24          4  46.68

Total observations: 24
Mean: 46.02 MM | Std: 14.18 MM


## TRAIN/TEST SPLIT

In [4]:
train = df.iloc[:20]  # Through 4Q23
test = df.iloc[20:]   # 1Q24-4Q24
print(f"\nTrain: {len(train)} quarters (1Q19-4Q23)")
print(f"Test:  {len(test)} quarters (1Q24-4Q24)")


Train: 20 quarters (1Q19-4Q23)
Test:  4 quarters (1Q24-4Q24)


## METHOD 1: LINEAR REGRESSION WITH TREND + SEASONALITY

In [5]:
print("\n" + "="*70)
print("METHOD 1: OLS Regression (Trend + Seasonal Dummies)")
print("="*70)

# Create features: trend + quarterly dummies
def make_regression_features(data):
    X = pd.DataFrame()
    X['trend'] = data['Period']
    X['Q1'] = (data['FiscalQtr'] == 1).astype(int)
    X['Q2'] = (data['FiscalQtr'] == 2).astype(int)
    X['Q3'] = (data['FiscalQtr'] == 3).astype(int)
    # Q4 is the baseline (omitted to avoid multicollinearity)
    return X

X_train = make_regression_features(train)
y_train = train['Sales'].values
X_test = make_regression_features(test)
y_test = test['Sales'].values

reg = LinearRegression()
reg.fit(X_train, y_train)

print(f"\nCoefficients:")
print(f"  Intercept:  {reg.intercept_:.4f}")
for name, coef in zip(X_train.columns, reg.coef_):
    print(f"  {name:10s}: {coef:.4f}")

y_pred_reg = reg.predict(X_test)
print(f"\nRegression equation: Sales = {reg.intercept_:.2f} + {reg.coef_[0]:.2f}*t + {reg.coef_[1]:.2f}*Q1 + {reg.coef_[2]:.2f}*Q2 + {reg.coef_[3]:.2f}*Q3")

print(f"\nForecasts vs Actual (Test Set):")
for i, (q, a, p) in enumerate(zip(test['Quarter'], y_test, y_pred_reg)):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_reg = mean_absolute_error(y_test, y_pred_reg)
mape_reg = np.mean(np.abs((y_test - y_pred_reg) / y_test)) * 100
print(f"\n  MAD:  {mae_reg:.2f}")
print(f"  MAPE: {mape_reg:.2f}%")

# Forecast 2025
future = pd.DataFrame({
    'Quarter': ['1Q25','2Q25','3Q25','4Q25'],
    'FiscalQtr': [1,2,3,4],
    'Period': [25,26,27,28]
})
X_future = make_regression_features(future)
forecast_2025_reg = reg.predict(X_future)
print(f"\n  2025 Forecasts: {dict(zip(future['Quarter'], np.round(forecast_2025_reg, 2)))}")


METHOD 1: OLS Regression (Trend + Seasonal Dummies)

Coefficients:
  Intercept:  19.6100
  trend     : 1.5915
  Q1        : 23.0885
  Q2        : 10.1890
  Q3        : -1.0365

Regression equation: Sales = 19.61 + 1.59*t + 23.09*Q1 + 10.19*Q2 + -1.04*Q3

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=76.12, Error=2.17
  2Q24: Actual=50.76, Forecast=64.81, Error=-14.05
  3Q24: Actual=41.03, Forecast=55.18, Error=-14.15
  4Q24: Actual=46.68, Forecast=57.81, Error=-11.13

  MAD:  10.37
  MAPE: 22.19%

  2025 Forecasts: {'1Q25': np.float64(82.49), '2Q25': np.float64(71.18), '3Q25': np.float64(61.54), '4Q25': np.float64(64.17)}


## METHOD 2: ARIMA-STYLE (Manual Implementation)

In [6]:
print("\n" + "="*70)
print("METHOD 2: AR(1) Model (Autoregressive - ARIMA Building Block)")
print("="*70)

# AR(1): Y_t = c + phi * Y_{t-1} + epsilon
# First deseasonalize
si = train.groupby('FiscalQtr')['Sales'].mean() / train['Sales'].mean()
print(f"\nSeasonality Indices: {dict(zip(si.index, np.round(si.values, 4)))}")

train_deseason = train.copy()
train_deseason['DesSales'] = train['Sales'] / train['FiscalQtr'].map(si)

# Fit AR(1) on deseasonalized data
y_des = train_deseason['DesSales'].values
y_lag = y_des[:-1].reshape(-1, 1)
y_cur = y_des[1:]

ar1 = LinearRegression()
ar1.fit(y_lag, y_cur)
c_ar = ar1.intercept_
phi_ar = ar1.coef_[0]
print(f"\nAR(1) model: Y_t = {c_ar:.4f} + {phi_ar:.4f} * Y_{{t-1}}")

# Forecast test period iteratively
last_des = y_des[-1]
ar1_forecasts_des = []
for i in range(4):
    next_val = c_ar + phi_ar * last_des
    ar1_forecasts_des.append(next_val)
    last_des = next_val

# Re-seasonalize
ar1_forecasts = [f * si[q] for f, q in zip(ar1_forecasts_des, [1,2,3,4])]

print(f"\nForecasts vs Actual (Test Set):")
for q, a, p in zip(test['Quarter'], y_test, ar1_forecasts):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_ar1 = mean_absolute_error(y_test, ar1_forecasts)
mape_ar1 = np.mean(np.abs((y_test - np.array(ar1_forecasts)) / y_test)) * 100
print(f"\n  MAD:  {mae_ar1:.2f}")
print(f"  MAPE: {mape_ar1:.2f}%")



METHOD 2: AR(1) Model (Autoregressive - ARIMA Building Block)

Seasonality Indices: {1: np.float64(1.2848), 2: np.float64(1.03), 3: np.float64(0.813), 4: np.float64(0.8722)}

AR(1) model: Y_t = 7.3886 + 0.8599 * Y_{t-1}

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=67.14, Error=11.15
  2Q24: Actual=50.76, Forecast=53.90, Error=-3.14
  3Q24: Actual=41.03, Forecast=42.59, Error=-1.56
  4Q24: Actual=46.68, Forecast=45.73, Error=0.95

  MAD:  4.20
  MAPE: 6.56%


## METHOD 3: SARIMA-STYLE (Seasonal AR with Differencing)

In [7]:
print("\n" + "="*70)
print("METHOD 3: Seasonal AR(1) with First Differencing (SARIMA-style)")
print("="*70)

# SARIMA(1,1,0)(1,0,0)[4] approximation
# Take first difference, then use both lag-1 and seasonal lag-4
sales = train['Sales'].values
diff_sales = np.diff(sales)  # first difference

# Features: lag-1 diff and seasonal (lag-4) level
X_sarima = []
y_sarima = []
for t in range(4, len(diff_sales)):
    X_sarima.append([diff_sales[t-1], sales[t-3]])  # AR(1) on diff + seasonal
    y_sarima.append(diff_sales[t])

X_sarima = np.array(X_sarima)
y_sarima = np.array(y_sarima)

sarima_model = LinearRegression()
sarima_model.fit(X_sarima, y_sarima)
print(f"\nSARIMA-style coefficients:")
print(f"  Intercept:       {sarima_model.intercept_:.4f}")
print(f"  AR(1) on diff:   {sarima_model.coef_[0]:.4f}")
print(f"  Seasonal lag-4:  {sarima_model.coef_[1]:.4f}")

# Forecast iteratively
all_sales = list(sales)
all_diffs = list(diff_sales)
sarima_forecasts = []
for i in range(4):
    x_new = np.array([[all_diffs[-1], all_sales[-3]]])
    diff_pred = sarima_model.predict(x_new)[0]
    sales_pred = all_sales[-1] + diff_pred
    sarima_forecasts.append(sales_pred)
    all_sales.append(sales_pred)
    all_diffs.append(diff_pred)

print(f"\nForecasts vs Actual (Test Set):")
for q, a, p in zip(test['Quarter'], y_test, sarima_forecasts):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_sarima = mean_absolute_error(y_test, sarima_forecasts)
mape_sarima = np.mean(np.abs((y_test - np.array(sarima_forecasts)) / y_test)) * 100
print(f"\n  MAD:  {mae_sarima:.2f}")
print(f"  MAPE: {mape_sarima:.2f}%")



METHOD 3: Seasonal AR(1) with First Differencing (SARIMA-style)

SARIMA-style coefficients:
  Intercept:       -23.4512
  AR(1) on diff:   -0.3003
  Seasonal lag-4:  0.5535

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=48.86, Error=29.43
  2Q24: Actual=50.76, Forecast=46.76, Error=4.00
  3Q24: Actual=41.03, Forecast=49.13, Error=-8.10
  4Q24: Actual=46.68, Forecast=52.02, Error=-5.34

  MAD:  11.72
  MAPE: 19.16%


## METHOD 4: RANDOM FOREST

In [8]:
print("\n" + "="*70)
print("METHOD 4: Random Forest Regression")
print("="*70)

def make_ml_features(data, all_sales_history):
    X = pd.DataFrame()
    X['trend'] = data['Period'].values
    X['Q1'] = (data['FiscalQtr'] == 1).astype(int).values
    X['Q2'] = (data['FiscalQtr'] == 2).astype(int).values
    X['Q3'] = (data['FiscalQtr'] == 3).astype(int).values
    # Lagged features
    lag1, lag4 = [], []
    for idx in data.index:
        p = data.loc[idx, 'Period']
        lag1.append(all_sales_history.get(p-1, np.nan))
        lag4.append(all_sales_history.get(p-4, np.nan))
    X['lag1'] = lag1
    X['lag4'] = lag4
    return X

# Build history dict
sales_dict = dict(zip(df['Period'], df['Sales']))

X_train_ml = make_ml_features(train, sales_dict)
X_train_ml = X_train_ml.dropna()
y_train_ml = train.loc[X_train_ml.index, 'Sales'].values

X_test_ml = make_ml_features(test, sales_dict)

rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=4)
rf.fit(X_train_ml, y_train_ml)

rf_pred = rf.predict(X_test_ml)
print(f"\nFeature Importances:")
for name, imp in zip(X_train_ml.columns, rf.feature_importances_):
    print(f"  {name:8s}: {imp:.4f}")

print(f"\nForecasts vs Actual (Test Set):")
for q, a, p in zip(test['Quarter'], y_test, rf_pred):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_rf = mean_absolute_error(y_test, rf_pred)
mape_rf = np.mean(np.abs((y_test - rf_pred) / y_test)) * 100
print(f"\n  MAD:  {mae_rf:.2f}")
print(f"  MAPE: {mape_rf:.2f}%")



METHOD 4: Random Forest Regression

Feature Importances:
  trend   : 0.0615
  Q1      : 0.0876
  Q2      : 0.0279
  Q3      : 0.0130
  lag1    : 0.0780
  lag4    : 0.7320

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=68.73, Error=9.56
  2Q24: Actual=50.76, Forecast=56.99, Error=-6.23
  3Q24: Actual=41.03, Forecast=44.26, Error=-3.23
  4Q24: Actual=46.68, Forecast=48.23, Error=-1.55

  MAD:  5.14
  MAPE: 8.92%


# METHOD 5: GRADIENT BOOSTING

In [9]:
print("\n" + "="*70)
print("METHOD 5: Gradient Boosting Regression")
print("="*70)

gb = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gb.fit(X_train_ml, y_train_ml)

gb_pred = gb.predict(X_test_ml)
print(f"\nForecasts vs Actual (Test Set):")
for q, a, p in zip(test['Quarter'], y_test, gb_pred):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_gb = mean_absolute_error(y_test, gb_pred)
mape_gb = np.mean(np.abs((y_test - gb_pred) / y_test)) * 100
print(f"\n  MAD:  {mae_gb:.2f}")
print(f"  MAPE: {mape_gb:.2f}%")



METHOD 5: Gradient Boosting Regression

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=71.66, Error=6.63
  2Q24: Actual=50.76, Forecast=50.46, Error=0.30
  3Q24: Actual=41.03, Forecast=40.50, Error=0.53
  4Q24: Actual=46.68, Forecast=44.52, Error=2.16

  MAD:  2.41
  MAPE: 3.74%


## METHOD 6: NEURAL NETWORK (MLP)

In [10]:
print("\n" + "="*70)
print("METHOD 6: Neural Network (Multi-Layer Perceptron)")
print("="*70)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_ml)
X_test_scaled = scaler.transform(X_test_ml)

mlp = MLPRegressor(hidden_layer_sizes=(16, 8), max_iter=2000,
                   activation='relu', solver='adam', random_state=42,
                   learning_rate_init=0.01, early_stopping=True)
mlp.fit(X_train_scaled, y_train_ml)

mlp_pred = mlp.predict(X_test_scaled)
print(f"\nNetwork architecture: Input(6) -> Hidden(16) -> Hidden(8) -> Output(1)")
print(f"Activation: ReLU | Optimizer: Adam | Epochs trained: {mlp.n_iter_}")

print(f"\nForecasts vs Actual (Test Set):")
for q, a, p in zip(test['Quarter'], y_test, mlp_pred):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_mlp = mean_absolute_error(y_test, mlp_pred)
mape_mlp = np.mean(np.abs((y_test - mlp_pred) / y_test)) * 100
print(f"\n  MAD:  {mae_mlp:.2f}")
print(f"  MAPE: {mape_mlp:.2f}%")



METHOD 6: Neural Network (Multi-Layer Perceptron)

Network architecture: Input(6) -> Hidden(16) -> Hidden(8) -> Output(1)
Activation: ReLU | Optimizer: Adam | Epochs trained: 98

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=88.58, Error=-10.29
  2Q24: Actual=50.76, Forecast=69.87, Error=-19.11
  3Q24: Actual=41.03, Forecast=58.59, Error=-17.56
  4Q24: Actual=46.68, Forecast=64.77, Error=-18.09

  MAD:  16.26
  MAPE: 33.08%


## METHOD 7: HOLT-WINTERS (Triple Exponential Smoothing)

In [11]:
print("\n" + "="*70)
print("METHOD 7: Holt-Winters (Triple Exponential Smoothing)")
print("="*70)

def holt_winters(series, season_len, alpha, beta, gamma, n_forecast=4):
    """Multiplicative Holt-Winters implementation."""
    n = len(series)
    # Initialize level and trend from first season
    level = np.mean(series[:season_len])
    trend = (np.mean(series[season_len:2*season_len]) - np.mean(series[:season_len])) / season_len
    # Initialize seasonal indices
    seasonal = [series[i] / np.mean(series[:season_len]) for i in range(season_len)]

    fitted = [None] * n
    for t in range(n):
        s = seasonal[t % season_len]
        if t == 0:
            fitted[t] = (level + trend) * s
        else:
            new_level = alpha * (series[t] / s) + (1 - alpha) * (level + trend)
            new_trend = beta * (new_level - level) + (1 - beta) * trend
            new_seasonal = gamma * (series[t] / (level + trend)) + (1 - gamma) * s
            level = new_level
            trend = new_trend
            seasonal[t % season_len] = new_seasonal
            fitted[t] = (level + trend) * seasonal[t % season_len]

    # Forecast future periods
    forecasts = []
    for k in range(1, n_forecast + 1):
        forecasts.append((level + k * trend) * seasonal[(n - 1 + k) % season_len])
    return fitted, forecasts

# Optimize parameters
def hw_loss(params, series, season_len):
    a, b, g = params
    if not (0 < a < 1 and 0 < b < 1 and 0 < g < 1):
        return 1e10
    try:
        fitted, _ = holt_winters(series, season_len, a, b, g, 0)
        errors = [abs(series[i] - fitted[i]) for i in range(season_len, len(series)) if fitted[i] is not None]
        return np.mean(errors)
    except:
        return 1e10

train_sales = train['Sales'].values
result = minimize(hw_loss, [0.5, 0.1, 0.3], args=(train_sales, 4),
                  method='Nelder-Mead', options={'maxiter': 5000})
opt_a, opt_b, opt_g = result.x

print(f"\nOptimal parameters: alpha={opt_a:.4f}, beta={opt_b:.4f}, gamma={opt_g:.4f}")

fitted_hw, forecast_hw = holt_winters(train_sales, 4, opt_a, opt_b, opt_g, 4)

print(f"\nForecasts vs Actual (Test Set):")
for q, a, p in zip(test['Quarter'], y_test, forecast_hw):
    print(f"  {q}: Actual={a:.2f}, Forecast={p:.2f}, Error={a-p:.2f}")

mae_hw = mean_absolute_error(y_test, forecast_hw)
mape_hw = np.mean(np.abs((y_test - np.array(forecast_hw)) / y_test)) * 100
print(f"\n  MAD:  {mae_hw:.2f}")
print(f"  MAPE: {mape_hw:.2f}%")



METHOD 7: Holt-Winters (Triple Exponential Smoothing)

Optimal parameters: alpha=0.9867, beta=0.1368, gamma=0.0000

Forecasts vs Actual (Test Set):
  1Q24: Actual=78.29, Forecast=63.40, Error=14.89
  2Q24: Actual=50.76, Forecast=60.80, Error=-10.04
  3Q24: Actual=41.03, Forecast=45.72, Error=-4.69
  4Q24: Actual=46.68, Forecast=47.87, Error=-1.19

  MAD:  7.70
  MAPE: 13.19%


# SUMMARY COMPARISON

In [12]:
print("\n" + "="*70)
print("MODEL COMPARISON SUMMARY")
print("="*70)

results = pd.DataFrame({
    'Method': ['OLS Regression', 'AR(1)', 'SARIMA-style', 'Random Forest',
               'Gradient Boosting', 'Neural Network (MLP)', 'Holt-Winters'],
    'MAD': [mae_reg, mae_ar1, mae_sarima, mae_rf, mae_gb, mae_mlp, mae_hw],
    'MAPE (%)': [mape_reg, mape_ar1, mape_sarima, mape_rf, mape_gb, mape_mlp, mape_hw]
})
results = results.sort_values('MAPE (%)').reset_index(drop=True)
results.index = results.index + 1
print(results.to_string())



MODEL COMPARISON SUMMARY
                 Method        MAD   MAPE (%)
1     Gradient Boosting   2.405287   3.744710
2                 AR(1)   4.197239   6.561034
3         Random Forest   5.142148   8.919404
4          Holt-Winters   7.701309  13.192856
5          SARIMA-style  11.716464  19.161255
6        OLS Regression  10.374000  22.192917
7  Neural Network (MLP)  16.260301  33.082344


# VISUALIZATION

In [14]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('iPhone Quarterly Sales: Advanced Forecasting Methods Comparison', fontsize=14, fontweight='bold')

quarters = list(df['Quarter'])
actual = list(df['Sales'])
forecast_quarters = list(test['Quarter'])

# Plot 1: Regression
ax = axes[0, 0]
ax.plot(quarters, actual, 'b-o', markersize=4, label='Actual', linewidth=1.5)
reg_full = reg.predict(make_regression_features(df))
ax.plot(quarters, reg_full, 'r--s', markersize=3, label='OLS Regression', linewidth=1)
ax.set_title(f'OLS Regression (MAPE={mape_reg:.1f}%)', fontsize=11)
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.set_ylabel('Sales (MM)')
ax.grid(alpha=0.3)

# Plot 2: AR(1) / SARIMA
ax = axes[0, 1]
ax.plot(quarters, actual, 'b-o', markersize=4, label='Actual', linewidth=1.5)
ax.plot(forecast_quarters, ar1_forecasts, 'g--^', markersize=5, label='AR(1)', linewidth=1.5)
ax.plot(forecast_quarters, sarima_forecasts, 'm--D', markersize=5, label='SARIMA-style', linewidth=1.5)
ax.set_title(f'Time Series Models', fontsize=11)
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.grid(alpha=0.3)

# Plot 3: ML Methods
ax = axes[1, 0]
ax.plot(quarters, actual, 'b-o', markersize=4, label='Actual', linewidth=1.5)
ax.plot(forecast_quarters, rf_pred, 'r--s', markersize=5, label=f'Random Forest', linewidth=1.5)
ax.plot(forecast_quarters, gb_pred, 'g--^', markersize=5, label=f'Gradient Boosting', linewidth=1.5)
ax.plot(forecast_quarters, mlp_pred, 'm--D', markersize=5, label=f'Neural Network', linewidth=1.5)
ax.set_title('Machine Learning Methods', fontsize=11)
ax.legend(fontsize=8)
ax.tick_params(axis='x', rotation=90, labelsize=6)
ax.set_ylabel('Sales (MM)')
ax.grid(alpha=0.3)

# Plot 4: Summary bar chart
ax = axes[1, 1]
methods_short = ['OLS\nReg', 'AR(1)', 'SARIMA', 'RF', 'GBM', 'MLP', 'HW']
mapes = [mape_reg, mape_ar1, mape_sarima, mape_rf, mape_gb, mape_mlp, mape_hw]
colors = ['#4472C4' if m == min(mapes) else '#A9A9A9' for m in mapes]
bars = ax.bar(methods_short, mapes, color=colors, edgecolor='white')
ax.set_title('MAPE Comparison (Lower is Better)', fontsize=11)
ax.set_ylabel('MAPE (%)')
for bar, m in zip(bars, mapes):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
            f'{m:.1f}%', ha='center', va='bottom', fontsize=8)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/home/forecasting_comparison.png', dpi=150, bbox_inches='tight')
print("\nChart saved to forecasting_comparison.png")


Chart saved to forecasting_comparison.png


# ADDITIONAL CHART: Feature Importance (Random Forest)

In [ ]:
fig2, ax2 = plt.subplots(figsize=(8, 4))
feat_imp = pd.Series(rf.feature_importances_, index=X_train_ml.columns).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=ax2, color='#4472C4')
ax2.set_title('Random Forest: Feature Importance', fontsize=12, fontweight='bold')
ax2.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('/home/feature_importance.png', dpi=150, bbox_inches='tight')
print("Feature importance chart saved.")